# MMAudio ile Ses Üretimi

**MMAudio** video ve/veya metin girdisinden ses üreten bir yapay zeka modelidir.

Bu notebook'ta üç mod desteklenir:
- **Video + Text → Audio** — Bir video yükleyip metin ile yönlendirin (en iyi kalite)
- **Video → Audio** — Sadece video verin, model sesi otomatik üretsin
- **Text → Audio** — Sadece metin açıklaması ile ses üretin

Tüm parametreler **Konfigürasyon** hücresinde toplanmıştır — farklı değerlerle deney yapabilirsiniz.

**Model:** `large_44k_v2` (44kHz, yüksek kalite) veya `small_16k` (16kHz, hızlı)  
**Lisans:** Model ağırlıkları CC-BY-NC 4.0 (yalnızca ticari olmayan kullanım)  
**GitHub:** https://github.com/hkchengrex/MMAudio

## GPU Ayarı

Bu notebook GPU gerektirir. T4 GPU (ücretsiz) yeterlidir.

1. Üst menüden **Runtime** → **Change runtime type** tıklayın
2. **Hardware accelerator** altından **T4 GPU** seçin
3. **Save** tıklayın

Aşağıdaki hücreyi çalıştırarak GPU'nun bağlı olduğunu doğrulayın.

In [ ]:
# GPU bagli mi kontrol et
import torch

if torch.cuda.is_available():
    # GPU adini ve bellek bilgisini yazdir
    gpu_name = torch.cuda.get_device_name(0)
    props = torch.cuda.get_device_properties(0)
    gpu_memory = getattr(props, 'total_memory', getattr(props, 'total_mem', 0)) / (1024**3)
    print(f"GPU bulundu: {gpu_name}")
    print(f"GPU bellegi: {gpu_memory:.1f} GB")
else:
    # GPU yoksa uyari ver
    print("UYARI: GPU bulunamadi!")
    print("Lutfen Runtime > Change runtime type > T4 GPU secin.")
    print("GPU olmadan model cok yavas calisir.")

## Konfigürasyon

Aşağıdaki hücrede tüm ayarlanabilir parametreler bir arada verilmiştir.  
Değerleri değiştirip tekrar çalıştırarak farklı sonuçları deneyebilirsiniz.

### Google Drive Ayarları

Videolarınızı Google Drive'da bir klasöre yükleyin. Notebook oradan okuyup sonuçları output klasörüne kaydeder.

| Parametre | Default | Açıklama |
|-----------|---------|----------|
| **DRIVE_VIDEO_FOLDER** | `"mmaudio-videos"` | Drive'daki video klasörü (MyDrive altında) |
| **DRIVE_OUTPUT_FOLDER** | `"mmaudio-outputs"` | Sonuçların kaydedileceği klasör |
| **BATCH_MODE** | `False` | `True` = tüm videoları sırayla işle |
| **VIDEO_INDEX** | `0` | `BATCH_MODE=False` ise hangi videoyu işle (listeden index) |

### Model Cache Ayarları

Model dosyaları (~2 GB) ilk çalıştırmada indirilir. Drive'da cache'leyerek sonraki seferlerde indirmeyi atlayabilirsiniz.

| Parametre | Default | Açıklama |
|-----------|---------|----------|
| **CACHE_MODELS_ON_DRIVE** | `True` | `True` = model dosyalarını Drive'da cachele (sonraki seferlerde indirme atlanır) |
| **DRIVE_MODEL_FOLDER** | `"mmaudio-models"` | Model cache klasörü (MyDrive altında) |

### Üretim Ayarları

Ses süresi otomatik olarak videonun süresine göre ayarlanır.

| Parametre | Default | Açıklama |
|-----------|---------|----------|
| **PROMPT** | — | Üretmek istediğiniz sesi tarif edin (İngilizce). Video modunda opsiyonel rehber |
| **NEGATIVE_PROMPT** | `""` | İstemediğiniz sesleri tarif edin (ör. `"music, speech"`) |
| **MODEL_NAME** | `"large_44k"` | Model seçimi (NSFW fine-tuned `"large_44k"` mimarisi) |
| **SEED** | `42` | Rastgele sayı tohumu. Aynı seed = aynı sonuç (tekrarlanabilirlik) |
| **NUM_STEPS** | `25` | Flow matching adım sayısı. Fazla = kaliteli ama yavaş, az = hızlı ama kaba |
| **INFERENCE_MODE** | `"euler"` | `"euler"` (sabit adım, hızlı) veya `"adaptive"` (otomatik adım, daha doğru) |
| **CFG_STRENGTH** | `4.5` | Rehberlik gücü. Yüksek = prompt'a daha sadık, düşük = daha serbest |

### Çıktı İsimlendirmesi

```
mmaudio-videos/dog_running.mp4  -->  mmaudio-outputs/dog_running_audio.mp4
mmaudio-videos/rain_window.mp4  -->  mmaudio-outputs/rain_window_audio.mp4
```

In [ ]:
# ============================================================
#  KONFIGURASYON - Tum ayarlari buradan degistirin
# ============================================================

# --- Google Drive ---
# Drive'daki video klasoru (MyDrive altinda)
DRIVE_VIDEO_FOLDER = "mmaudio-videos"

# Sonuclarin kaydedilecegi klasor (MyDrive altinda)
DRIVE_OUTPUT_FOLDER = "mmaudio-outputs"

# True = tum videolari sirayla isle, False = sadece VIDEO_INDEX'teki videoyu isle
BATCH_MODE = False

# BATCH_MODE=False ise hangi videoyu isle (0'dan baslar)
VIDEO_INDEX = 0

# --- Model Cache ---
# True = model dosyalarini Drive'da cachele (sonraki seferlerde indirme atlanir)
# False = her seferde HuggingFace'den indir
CACHE_MODELS_ON_DRIVE = True

# Model cache klasoru (MyDrive altinda)
DRIVE_MODEL_FOLDER = "mmaudio-models"

# --- Prompt ---
# Sesi tarif edin (Ingilizce). Video modunda opsiyonel rehber olarak kullanilir
PROMPT = ""

# Istemediginiz sesleri tarif edin (bos birakabilirsiniz)
# Ornek: "music, speech, noise"
NEGATIVE_PROMPT = ""

# Rastgele sayi tohumu - ayni seed ayni sonucu verir (tekrarlanabilirlik)
SEED = 42

# --- Model ---
# NSFW fine-tuned model "large_44k" (v1) mimarisini kullanir, degistirmeyin
MODEL_NAME = "large_44k"

# --- Flow Matching ---
# Adim sayisi: fazla = kaliteli ama yavas, az = hizli ama kaba
# Onerilen aralik: 10-50, default: 25
NUM_STEPS = 25

# Cozucu modu:
#   "euler"    = sabit adim, hizli ve tahmin edilebilir
#   "adaptive" = otomatik adim boyutu, daha dogru (NUM_STEPS gormezden gelinir)
INFERENCE_MODE = "euler"

# --- Rehberlik ---
# Classifier-free guidance gucu
# Yuksek = prompt'a daha sadik, dusuk = daha serbest
# Onerilen aralik: 1.0-10.0, default: 4.5
CFG_STRENGTH = 4.5

# ============================================================
print("=== Konfigurasyon ===")
print(f"  Drive Video:     {DRIVE_VIDEO_FOLDER}/")
print(f"  Drive Output:    {DRIVE_OUTPUT_FOLDER}/")
print(f"  Batch Mode:      {BATCH_MODE}")
if not BATCH_MODE:
    print(f"  Video Index:     {VIDEO_INDEX}")
if CACHE_MODELS_ON_DRIVE:
    print(f"  Model Cache:     Drive/{DRIVE_MODEL_FOLDER}/ (acik)")
else:
    print(f"  Model Cache:     kapali (her seferde indirir)")
print(f"  Prompt:          {PROMPT or '(bos)'}")
print(f"  Negatif Prompt:  {NEGATIVE_PROMPT or '(yok)'}")
print(f"  Sure:            videonun suresi")
print(f"  Seed:            {SEED}")
print(f"  Model:           {MODEL_NAME} (NSFW fine-tuned)")
print(f"  Adim Sayisi:     {NUM_STEPS}")
print(f"  Cozucu:          {INFERENCE_MODE}")
print(f"  CFG Gucu:        {CFG_STRENGTH}")
print("=" * 25)

## Google Drive Bağlantısı

Önce Drive izinlerini alıp video klasörünü tarayalım — böylece sorun varsa uzun kurulum adımlarını beklemeye gerek kalmaz.  
İlk çalıştırmada Google hesabınıza erişim izni isteyecektir.

In [ ]:
# Google Drive'i bagla ve video klasorunu tara
from google.colab import drive
import os

drive.mount('/content/drive')

drive_video_path = f"/content/drive/MyDrive/{DRIVE_VIDEO_FOLDER}"
drive_output_path = f"/content/drive/MyDrive/{DRIVE_OUTPUT_FOLDER}"

# Output klasorunu olustur (yoksa)
os.makedirs(drive_output_path, exist_ok=True)

# Video dosyalarini listele
video_extensions = ('.mp4', '.avi', '.mov', '.mkv', '.webm')
videos = sorted([
    f for f in os.listdir(drive_video_path)
    if f.lower().endswith(video_extensions)
])

print(f"Klasor: Drive/{DRIVE_VIDEO_FOLDER}/")
print(f"Bulunan videolar ({len(videos)}):")
for i, v in enumerate(videos):
    size_mb = os.path.getsize(f"{drive_video_path}/{v}") / (1024 * 1024)
    print(f"  [{i}] {v} ({size_mb:.1f} MB)")

print()
if BATCH_MODE:
    print(f"BATCH MODE: Tum {len(videos)} video islenecek")
else:
    print(f"Secilen video: [{VIDEO_INDEX}] {videos[VIDEO_INDEX]}")

## Kurulum

Şimdi MMAudio reposunu klonlayıp gerekli kütüphaneleri yükleyeceğiz.  
Bu adım birkaç dakika sürebilir.

In [ ]:
# MMAudio reposunu klonla ve tum bagimliliklari kur
!git clone https://github.com/hkchengrex/MMAudio.git
%cd MMAudio
!pip install -e . -q

# Pip cache'i temizle (RAM tasarrufu)
!pip cache purge -q

print("\nKurulum tamamlandi!")

In [ ]:
# Model agirliklarini indir (Drive cache destekli)
import os
import shutil
from pathlib import Path
from mmaudio.eval_utils import all_model_cfg
from huggingface_hub import hf_hub_download

model_cfg = all_model_cfg[MODEL_NAME]
nsfw_filename = "mmaudio_large_44k_nsfw_gold_8.5k_final_fp16.safetensors"

if CACHE_MODELS_ON_DRIVE:
    drive_model_path = f"/content/drive/MyDrive/{DRIVE_MODEL_FOLDER}"
    local_model_dir = "/content/model_cache"
    os.makedirs(drive_model_path, exist_ok=True)
    os.makedirs(local_model_dir, exist_ok=True)

    cache_marker = f"{drive_model_path}/.cache_complete"

    if os.path.exists(cache_marker):
        # === CACHE HIT: Drive'dan yukle, indirme atla ===
        print(f"Drive/{DRIVE_MODEL_FOLDER}/ icinde model cache bulundu!")
        print("Modeller Drive'dan local'e kopyalaniyor (indirme atlanıyor)...\n")

        for fname in os.listdir(drive_model_path):
            if fname.startswith('.'):
                continue
            src = f"{drive_model_path}/{fname}"
            dst = f"{local_model_dir}/{fname}"
            if os.path.isfile(src):
                size_mb = os.path.getsize(src) / (1024**2)
                print(f"  {fname} ({size_mb:.0f} MB)")
                shutil.copy2(src, dst)

        # model_cfg path'lerini local kopyalara yonlendir
        model_cfg.vae_path = f"{local_model_dir}/vae.pth"
        model_cfg.synchformer_ckpt = f"{local_model_dir}/synchformer.pth"
        bigvgan_local = f"{local_model_dir}/bigvgan_16k.pth"
        if os.path.exists(bigvgan_local):
            model_cfg.bigvgan_16k_path = bigvgan_local

        nsfw_model_path = f"{local_model_dir}/{nsfw_filename}"
        print("\nModeller cache'den yuklendi! (indirme atlanildi)")

    else:
        # === CACHE MISS: Indir + Drive'a kaydet ===
        print("Drive'da model cache bulunamadi.")
        print("Modeller indiriliyor (ilk sefer, ~3 dk)...\n")

        # Standart model dosyalari (VAE, synchformer, bigvgan)
        model_cfg.download_if_needed()

        # NSFW fine-tuned model
        nsfw_model_path = hf_hub_download(
            repo_id="phazei/NSFW_MMaudio",
            filename=nsfw_filename,
        )
        print(f"NSFW model indirildi: {nsfw_model_path}")

        # Drive'a kaydet (sonraki seferde hizli yuklensin)
        print(f"\nModeller Drive/{DRIVE_MODEL_FOLDER}/ klasorune kaydediliyor...")
        files_to_cache = [
            ('vae.pth', str(model_cfg.vae_path)),
            ('synchformer.pth', str(model_cfg.synchformer_ckpt)),
            (nsfw_filename, nsfw_model_path),
        ]
        if hasattr(model_cfg, 'bigvgan_16k_path') and model_cfg.bigvgan_16k_path:
            files_to_cache.append(('bigvgan_16k.pth', str(model_cfg.bigvgan_16k_path)))

        total_mb = 0
        for cache_name, src_path in files_to_cache:
            if src_path and os.path.exists(src_path):
                size_mb = os.path.getsize(src_path) / (1024**2)
                total_mb += size_mb
                print(f"  {cache_name} ({size_mb:.0f} MB)")
                shutil.copy2(src_path, f"{drive_model_path}/{cache_name}")

        Path(cache_marker).touch()
        print(f"Cache kaydedildi! (toplam: {total_mb:.0f} MB)")
        print("Sonraki calistirmada indirme atlanacak.")

else:
    # Cache kapali - her seferde indir
    print("Model indiriliyor...")
    model_cfg.download_if_needed()
    nsfw_model_path = hf_hub_download(
        repo_id="phazei/NSFW_MMaudio",
        filename=nsfw_filename,
    )
    print(f"NSFW model indirildi: {nsfw_model_path}")

# Gecici tensorleri temizle
import torch
torch.cuda.empty_cache()
print(f"\nModel(ler) hazir! (NSFW FP16)")

## Model Yükleme

Model ağırlıklarını GPU'ya yükler. Bu hücre bir kez çalıştırılır.  
Farklı ayarlarla tekrar denemek için sadece **Ses Üretimi** hücresini tekrar çalıştırın.

In [ ]:
# Model yukle (bir kez calistirin)
import gc
import psutil
import torch
import torchaudio
from safetensors.torch import load_file
from mmaudio.eval_utils import setup_eval_logging, generate
from mmaudio.model.networks import get_my_mmaudio
from mmaudio.model.flow_matching import FlowMatching
from mmaudio.model.utils.features_utils import FeaturesUtils

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
setup_eval_logging()

device = 'cuda'
dtype = torch.float16  # T4 GPU bfloat16 desteklemez

def mem_status(label):
    ram = psutil.virtual_memory()
    gpu_used = torch.cuda.memory_allocated() / (1024**3)
    gpu_reserved = torch.cuda.memory_reserved() / (1024**3)
    print(f"[{label}] RAM: {ram.used/(1024**3):.1f}/{ram.total/(1024**3):.1f} GB (bos: {ram.available/(1024**3):.1f}) | GPU: {gpu_used:.1f} GB (reserved: {gpu_reserved:.1f})")

mem_status("baslangic")

# Agirliklari dogrudan GPU'ya yukle (CPU RAM kullanmadan)
print("Net olusturuluyor...")
net = get_my_mmaudio(model_cfg.model_name).to(device, dtype).eval()

# NSFW FP16 safetensors agirliklarini yukle
weights = load_file(nsfw_model_path, device=device)
net.load_weights(weights)
del weights
gc.collect()
torch.cuda.empty_cache()
mem_status("net GPU'da")

print("FeaturesUtils olusturuluyor...")
feature_utils = FeaturesUtils(
    tod_vae_ckpt=model_cfg.vae_path,
    synchformer_ckpt=model_cfg.synchformer_ckpt,
    enable_conditions=True,
    mode=model_cfg.mode,
    bigvgan_vocoder_ckpt=model_cfg.bigvgan_16k_path,
    need_vae_encoder=False
)   
mem_status("FeaturesUtils CPU'da olusturuldu")

feature_utils = feature_utils.to(device, dtype).eval()
gc.collect()
torch.cuda.empty_cache()
mem_status("FeaturesUtils GPU'ya tasindi")

print("Model hazir! (NSFW FP16)")

## Ses Üretimi

Videolardan ses üretip, ffmpeg ile birleştirip Drive'a kaydeder.  
Ayarları değiştirdikten sonra bu hücreyi tekrar çalıştırmanız yeterlidir.

In [ ]:
# Ses uret + video ile birlestir + Drive'a kaydet
import os
import shutil
import subprocess
from pathlib import Path
from mmaudio.model.flow_matching import FlowMatching
from mmaudio.eval_utils import load_video

# Flow matching ayarlari
fm = FlowMatching(min_sigma=0, inference_mode=INFERENCE_MODE, num_steps=NUM_STEPS)
rng = torch.Generator(device=device)
seq_cfg = model_cfg.seq_cfg

# Islenecek videolari belirle
video_list = videos if BATCH_MODE else [videos[VIDEO_INDEX]]

print(f"Islenecek video sayisi: {len(video_list)}")
print("=" * 50)

mem_status("ses uretimi baslangici")

for i, video_name in enumerate(video_list):
    stem = os.path.splitext(video_name)[0]
    print(f"\n[{i+1}/{len(video_list)}] {video_name}")

    # 1. Drive'dan local'e kopyala
    local_video_orig = f"/content/{video_name}"
    shutil.copy2(f"{drive_video_path}/{video_name}", local_video_orig)
    print("  Video local'e kopyalandi")

    # 2. 720p'ye kucult (MMAudio 224x224 frame cikarir, 4K gereksiz RAM harcar)
    local_video = f"/content/{stem}_720p.mp4"
    resize_result = subprocess.run([
        'ffmpeg', '-y', '-i', local_video_orig,
        '-vf', 'scale=-2:720',
        '-c:v', 'libx264', '-preset', 'ultrafast', '-crf', '23',
        '-an', local_video
    ], capture_output=True, text=True)

    if resize_result.returncode == 0:
        orig_mb = os.path.getsize(local_video_orig) / (1024 * 1024)
        new_mb = os.path.getsize(local_video) / (1024 * 1024)
        print(f"  720p'ye kucultuldu ({orig_mb:.0f} MB -> {new_mb:.0f} MB)")
    else:
        print("  Kucultme basarisiz, orijinal kullaniliyor")
        local_video = local_video_orig

    mem_status("video hazir")

    # 3. Videonun gercek suresini olc (ffprobe)
    probe = subprocess.run(
        ['ffprobe', '-v', 'error', '-show_entries', 'format=duration',
         '-of', 'default=noprint_wrappers=1:nokey=1', local_video],
        capture_output=True, text=True
    )
    video_duration = float(probe.stdout.strip())
    print(f"  Video suresi: {video_duration:.2f} saniye")

    # 4. Sequence uzunluklarini videoya gore ayarla
    seq_cfg.duration = video_duration
    net.update_seq_lengths(seq_cfg.latent_seq_len, seq_cfg.clip_seq_len, seq_cfg.sync_seq_len)

    # 5. Video'yu yukle (clip + sync frameleri cikar)
    video_info = load_video(Path(local_video), video_duration)
    mem_status("load_video sonrasi")

    clip_v = video_info.clip_frames.unsqueeze(0).to(device, dtype)
    sync_v = video_info.sync_frames.unsqueeze(0).to(device, dtype)
    del video_info
    gc.collect()
    print(f"  Video yuklendi")
    mem_status("frameler GPU'da")

    # 6. Ses uret
    rng.manual_seed(SEED)
    print(f"  Ses uretiliyor... (prompt: '{PROMPT or '(bos)'}')")
    with torch.inference_mode():
        audios = generate(
            clip_video=clip_v,
            sync_video=sync_v,
            text=[PROMPT] if PROMPT else [''],
            negative_text=[NEGATIVE_PROMPT],
            feature_utils=feature_utils,
            net=net,
            fm=fm,
            rng=rng,
            cfg_strength=CFG_STRENGTH,
        )
    mem_status("generate sonrasi")

    # 7. Sesi kaydet
    audio = audios.float().cpu()[0]
    local_audio = f"/content/{stem}_audio.flac"
    torchaudio.save(local_audio, audio, seq_cfg.sampling_rate)
    print("  Ses uretildi")

    # 8. ffmpeg ile ORIJINAL video + ses birlestir (cikti orijinal kalitede kalir)
    local_output = f"/content/{stem}_audio.mp4"
    result = subprocess.run([
        'ffmpeg', '-y',
        '-i', local_video_orig,
        '-i', local_audio,
        '-c:v', 'copy',
        '-c:a', 'aac',
        '-shortest',
        local_output
    ], capture_output=True, text=True)

    if result.returncode != 0:
        print(f"  HATA (ffmpeg): {result.stderr[-200:]}")
        continue

    print("  Video + ses birlestirildi")

    # 9. Drive output klasorune kaydet
    output_name = f"{stem}_audio.mp4"
    shutil.copy2(local_output, f"{drive_output_path}/{output_name}")
    print(f"  -> Drive/{DRIVE_OUTPUT_FOLDER}/{output_name}")

    # 10. Local dosyalari temizle
    for f in [local_video, local_video_orig, local_audio, local_output]:
        if os.path.exists(f):
            os.remove(f)

    del clip_v, sync_v, audios, audio
    torch.cuda.empty_cache()
    gc.collect()
    mem_status(f"video {i+1} temizlendi")

print("\n" + "=" * 50)
print(f"Tamamlandi! {len(video_list)} video islendi.")

# Drive'a yazilan dosyalarin sync olmasini garanti et
print("Drive'a senkronize ediliyor...")
drive.flush_and_unmount()
print("Drive senkronizasyonu tamamlandi!")
print(f"Sonuclar: Drive/{DRIVE_OUTPUT_FOLDER}/")